# Serve Qwen3.8-27B on Kaggle free tier

Turns a free Kaggle GPU session into an **OpenAI-compatible endpoint** on a public HTTPS URL,
so the agent running on your laptop can call it.

## Sidebar setup, once

1. **Accelerator** = `GPU T4 x2`
2. **Internet** = `On`
3. **Add-ons -> Secrets** = add `HF_TOKEN` and toggle it on. Optional: the LLM needs no token,
   but the rest of the pipeline pulls from the Hub.

## How to use it

| Run | `BUILD_CACHE` | What you do |
|---|---|---|
| First | `True` | **Run All**, then save (see below), then **Stop Session** |
| Every time after | `False` | **Add Input -> Notebook Output ->** this notebook, set `CACHE`, **Run All** |

The first run pulls 18 GB. Later runs skip that entirely.

### Saving the cache: use Quick Save, not Save & Run All

The build run ends by raising `RuntimeError("Cache build finished...")`. **That is the success
signal, not a crash.** Jupyter's Run All ignores `SystemExit`, so a raised exception is the only
thing that reliably stops it from continuing into the serving cells.

Then:

1. **Save Version -> Quick Save**
2. **Advanced Settings -> tick "Save output for this version"**
3. Save, then open the saved version's Output tab and **confirm it shows ~18 GB**

**Do not use Save & Run All here.** It re-executes the notebook in a fresh container, re-pulling all
18 GB and spending 15 to 25 minutes of GPU quota to rebuild a cache that is already on disk. Only
fall back to it if Quick Save's output comes out empty.

### What is in the cache, and what is not

| | Size | Cached |
|---|---|---|
| Model store | ~18 GB | **Yes** |
| Ollama binary and bundled CUDA libs | ~3 GB extracted | No, re-fetched each run |

Kaggle caps notebook output at 20 GB, so both do not fit. Ollama is the right one to drop: it
downloads at ~380 MB/s and extracts in about **13 seconds total**, whereas the model pull takes
minutes. The cache spends its 20 GB on the part that is actually slow.

## Stopping

**Stop Session** in the sidebar. Nothing else stops the quota meter. See the table at the bottom.

## Why Ollama, and not vLLM or llama-cpp-python

This was checked against release dates, not assumed.

**vLLM is wrong for this hardware.** vLLM 0.28 wants torch 2.13 and the CUDA 13 stack, so on Kaggle it
uninstalls and replaces torch, torchvision, triton, cudnn, cublas and roughly thirty other packages.
Worse, Turing (the T4, compute capability 7.5) is a neglected path in vLLM, with open reports of
*exactly* this configuration, a 27B Qwen on a T4, hanging indefinitely.

**llama-cpp-python cannot load this model.** The prebuilt CUDA wheel index tops out at **0.3.19,
released March 2026**. Qwen3.8-27B was released **13-14 August 2026** and registers a new `qwen35`
architecture; builds older than that **refuse the file outright**. Upstream llama-cpp-python is at
0.3.35 (17 Aug 2026) and would work, but has no prebuilt CUDA wheel, meaning a ~30 minute CMake
compile on every cache rebuild.

**Ollama fits.** Qwen3.8-27B support landed in **v0.32.12**, with follow-up fixes in v0.32.13 and
v0.32.14 for Qwen3.8 system-message handling. We pin v0.33.2 (27 Aug 2026), comfortably past all of it.

| | vLLM | llama-cpp-python | **Ollama** |
|---|---|---|---|
| Supports Qwen3.8 arch | Yes | **No** (wheel is pre-release) | **Yes**, v0.32.12+ |
| Turing / T4 | Neglected, documented hangs | Good | **Supported, cc 5.0+** |
| Touches Kaggle's CUDA stack | Replaces it | Links against it | **Ignores it, bundles its own** |
| Install | ~4 GB of wheels | One wheel + compile risk | **One 1.3 GB tarball** |
| Model size | 19.5 GB AWQ | 16.5 GB GGUF | **18 GB, Ollama's own build** |

The third row is the one that matters most. Ollama ships its own CUDA runtime, so whatever CUDA
version Kaggle's image happens to be on this month is **irrelevant**. Only the NVIDIA driver
matters, and that is fixed by the host.

### Three footguns this notebook works around

- `https://ollama.com/download/ollama-linux-amd64.tgz` **404s as of v0.33.2.** The asset was renamed
  to `ollama-linux-amd64.tar.zst`. Most tutorials online are stale.
- **Ollama has no authentication.** A Cloudflare tunnel would put an open 27B endpoint on the public
  internet. A small bearer-token proxy sits in front of it.
- **Ollama cannot import Unsloth "UD" GGUFs.** `ollama create` from `Qwen3.8-27B-UD-Q4_K_M.gguf` dies
  with `failed to validate GGUF with llama-quantize without compatibility patches`. UD quants vary the
  bit width per layer and Ollama's import validation gate rejects that; the same error is open upstream
  as [ollama#17279](https://github.com/ollama/ollama/issues/17279), with no bypass flag. We pull
  `qwen3.8:27b` from Ollama's registry instead, which has no import step and so no gate.

In [ ]:
# ============================================================
#  CONFIG - the only cell you edit
# ============================================================

BUILD_CACHE = False    # True for the one-time cache build. False every run after.

# Kaggle mounts a notebook output at a nested path, not a flat one. It looks
# like /kaggle/input/notebooks/<your-user>/<notebook-slug>. Copy the exact path
# from the Input panel. If you get it wrong, the auto-detect below rescues you.
CACHE = "/kaggle/input/notebooks/CHANGE-ME/CHANGE-ME"

RUN_MINUTES = 120      # auto-stop the model after this long
N_CTX       = 8192     # context window. Lower to 4096 if memory is tight.

OLLAMA_VER  = "v0.33.2"        # pinned. Qwen3.8 needs >= v0.32.12
MODEL_TAG   = "qwen3.8:27b"    # Ollama's own build, 18 GB
MODEL_NAME  = "qwen"           # our derived tag; the name clients pass as "model"

# Faster option, worth measuring at your next cache rebuild:
#   MODEL_TAG = "qwen3.8:27b-mtp-q4_K_M"    # also 18 GB, fits the same cache
#
# MTP is multi-token prediction: a small draft head proposes several tokens and
# the full model verifies them in one pass, so you get more than one token per
# read of the weights. Verification makes the output mathematically identical to
# normal decoding, so this is a free speedup, not a quality tradeoff. Independent
# reports put it at +33 to 39% on consumer GPUs for this exact model.
#
# UNVERIFIED HERE: Ollama's MTP support landed on the MLX runner first and the
# CUDA path was still being validated. Measure it, do not assume it.
#
# Other tags, all dense 27B: 27b-q8_0 (30 GB, too big for the cache),
# 27b-nvfp4 (18 GB, needs Blackwell), 27b-mlx* (Apple only), 27b-bf16 (56 GB).
# There is no MoE variant of Qwen3.8, so no cheap win from sparse activation.

import os, sys, time, datetime, subprocess, shutil, json
T0 = time.time()

def step(msg):
    # Timestamped progress, so a long pause is visibly not a hang.
    print(f"[{datetime.datetime.now():%H:%M:%S}  +{time.time()-T0:5.0f}s]  {msg}", flush=True)

def sh(cmd, **kw):
    # Run a command, stream its output, raise on failure.
    return subprocess.run(cmd, check=True, **kw)

# --- HF token from Kaggle Secrets -------------------------------------------
# Never paste a token into a cell. Add-ons -> Secrets -> HF_TOKEN, then attach
# it to this notebook with the toggle.
#
# The LLM comes from Ollama's registry and needs no token. This is wired up for
# the rest of the pipeline (Chronos-Bolt, the sentence embedding model), which
# are pulled from the Hub.
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    step(f"HF_TOKEN loaded from Kaggle Secrets ({len(HF_TOKEN)} chars, not printed)")
except Exception as e:
    step(f"HF_TOKEN not available ({type(e).__name__}), continuing without it")
    step("  Not needed to serve the LLM. To add: Add-ons -> Secrets -> HF_TOKEN,")
    step("  then toggle it on for this notebook.")

# --- paths -------------------------------------------------------------------
# Ollama itself goes to /kaggle/tmp (~60 GB, not persisted, re-downloaded each run).
# Only the model store goes to /kaggle/working, which is capped at 20 GB.
OLLAMA_HOME = "/kaggle/tmp/ollama"
STORE = "/kaggle/working/ollama-models" if BUILD_CACHE else "/kaggle/tmp/ollama-models"

CACHE_DONE = False

# Auto-detect the cache if CACHE is wrong. Kaggle's mount path for a notebook
# output is nested and easy to mistype, and the cost of getting it wrong is an
# unnecessary 18 GB pull. Bounded globs, not a recursive walk.
if not BUILD_CACHE and not os.path.isdir(f"{CACHE}/ollama-models/manifests"):
    import glob
    hits = (glob.glob("/kaggle/input/*/ollama-models/manifests")
            + glob.glob("/kaggle/input/*/*/ollama-models/manifests")
            + glob.glob("/kaggle/input/*/*/*/ollama-models/manifests"))
    if hits:
        CACHE = hits[0].rsplit("/ollama-models/", 1)[0]
        step(f"CACHE was wrong; auto-detected it at {CACHE}")
    else:
        step("no cached model store found anywhere under /kaggle/input")
        step("  attached inputs: " + str(sorted(os.listdir("/kaggle/input"))
                                          if os.path.isdir("/kaggle/input") else "none"))

HAVE_CACHE = (not BUILD_CACHE) and os.path.isdir(f"{CACHE}/ollama-models/manifests")

step("MODE: " + ("BUILD CACHE (one-time)" if BUILD_CACHE
                 else ("serve from cache" if HAVE_CACHE else "serve, NO CACHE - will pull 18 GB")))
if not BUILD_CACHE and not HAVE_CACHE:
    step(f"  nothing found at {CACHE}/ollama-models/")
    step("  Add Input -> Notebook Output -> this notebook, and check CACHE above.")

In [ ]:
# ============================================================
#  1. Environment probe. Confirms the assumptions above hold.
# ============================================================

step("GPU")
gpu = subprocess.run(["nvidia-smi",
    "--query-gpu=index,name,memory.total,compute_cap,driver_version",
    "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(gpu)

n_gpu = len([l for l in gpu.splitlines() if l.strip()])
assert n_gpu >= 1, "No GPU. Sidebar -> Accelerator -> GPU T4 x2, then Run All again."
if n_gpu < 2:
    print(f"\n  WARNING: {n_gpu} GPU visible. An 18 GB model does not fit one 15.6 GB T4.")
    print("  Sidebar -> Accelerator -> 'GPU T4 x2'.")

step("driver / CUDA")
# 'CUDA Version' here is the MAXIMUM the driver supports, not what is installed.
# It is the only CUDA number that matters to us: Ollama bundles its own runtime.
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout.splitlines()[2])

step("python / disk")
print("  python  :", sys.version.split()[0])
print("  ", subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/tmp"],
                           capture_output=True, text=True).stdout.replace("\n", "\n   "))
print("  zstd    :", shutil.which("zstd") or "not on PATH (python fallback will be used)")

In [ ]:
# ============================================================
#  2. Install Ollama and start the server.
# ============================================================

# NOTE: https://ollama.com/download/ollama-linux-amd64.tgz returns 404 as of
# v0.33.2 - the asset was renamed to .tar.zst. Most guides online are stale.
URL = (f"https://github.com/ollama/ollama/releases/download/{OLLAMA_VER}"
       f"/ollama-linux-amd64.tar.zst")
TARBALL = "/kaggle/tmp/ollama.tar.zst"

os.makedirs("/kaggle/tmp", exist_ok=True)
os.makedirs(OLLAMA_HOME, exist_ok=True)

if not os.path.exists(f"{OLLAMA_HOME}/bin/ollama"):
    step(f"downloading Ollama {OLLAMA_VER} (~1.3 GB)")
    sh(["wget", "--progress=bar:force:noscroll", "-O", TARBALL, URL])

    # Kaggle's image has no unzstd binary, so this normally takes the python
    # path. Checking first keeps a confusing tar error out of the log.
    step("extracting")
    if shutil.which("unzstd"):
        sh(["tar", "--use-compress-program=unzstd", "-xf", TARBALL, "-C", OLLAMA_HOME])
    else:
        step("  no unzstd binary; using python zstandard (expected on Kaggle)")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zstandard"], check=True)
        import zstandard, tarfile
        with open(TARBALL, "rb") as f, zstandard.ZstdDecompressor().stream_reader(f) as r:
            with tarfile.open(fileobj=r, mode="r|") as t:
                t.extractall(OLLAMA_HOME, filter="tar")
    os.remove(TARBALL)

OLLAMA = f"{OLLAMA_HOME}/bin/ollama"
os.chmod(OLLAMA, 0o755)
# 'ollama --version' warns it cannot reach a server, since serve is not up yet.
# Harmless; keep only the version line.
_v = subprocess.run([OLLAMA, "--version"], capture_output=True, text=True).stdout
step("installed: " + next((l for l in _v.splitlines() if "version is" in l), _v.strip()))

# --- wire the cached model store --------------------------------------------
# /kaggle/input is read-only and Ollama needs to write. Blobs are content
# addressed and immutable, so we symlink those and copy only the tiny manifests.
os.makedirs(f"{STORE}/blobs", exist_ok=True)
if HAVE_CACHE:
    step("linking cached model store (symlinked blobs, no 18 GB copy)")
    src = f"{CACHE}/ollama-models"
    if os.path.isdir(f"{STORE}/manifests"):
        shutil.rmtree(f"{STORE}/manifests")
    shutil.copytree(f"{src}/manifests", f"{STORE}/manifests")
    linked = 0
    for b in os.listdir(f"{src}/blobs"):
        dst = f"{STORE}/blobs/{b}"
        if not os.path.lexists(dst):
            os.symlink(f"{src}/blobs/{b}", dst); linked += 1

    # Verify the targets actually resolve. A cache carrying manifests but no
    # blobs is the failure mode to catch here: Ollama would start happily,
    # report "total blobs: 0", then die minutes later on the first request
    # with the misleading '"qwen" does not support chat'.
    #
    # It happens when a Quick Save is taken from a SERVING session, where
    # /kaggle/working holds only log files, and that near-empty version
    # becomes the attached output. Only ever save from a BUILD_CACHE run.
    usable = [b for b in os.listdir(f"{STORE}/blobs")
              if os.path.exists(f"{STORE}/blobs/{b}")]      # follows symlinks
    biggest = max((os.path.getsize(f"{STORE}/blobs/{b}") for b in usable), default=0)
    step(f"  {linked} linked, {len(usable)} resolve, largest {biggest/2**30:.2f} GiB")

    if len(usable) < 2 or biggest < 2 * 2**30:
        step("  CACHE IS UNUSABLE: manifests present but the weights are missing.")
        step("  Falling back to a fresh pull so this run still works.")
        step("  To fix it permanently, redo the build run and Quick Save from")
        step("  a BUILD_CACHE=True session only.")
        shutil.rmtree(STORE, ignore_errors=True)
        os.makedirs(f"{STORE}/blobs", exist_ok=True)
        HAVE_CACHE = False        # cell 3 will pull

# --- start the server --------------------------------------------------------
env = {**os.environ,
       "OLLAMA_HOME": OLLAMA_HOME,
       "OLLAMA_MODELS": STORE,
       "OLLAMA_HOST": "127.0.0.1:11434",
       "OLLAMA_KEEP_ALIVE": "24h",        # default is 5m; an unload means a slow reload
       "OLLAMA_CONTEXT_LENGTH": str(N_CTX),
       "OLLAMA_FLASH_ATTENTION": "1"}

SRV_LOG = "/kaggle/working/ollama-serve.log"
serve = subprocess.Popen([OLLAMA, "serve"], env=env,
                         stdout=open(SRV_LOG, "w"), stderr=subprocess.STDOUT)

import urllib.request, urllib.error
for i in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=3)
        step(f"ollama serve up, pid {serve.pid}"); break
    except Exception:
        time.sleep(2)
else:
    print("".join(open(SRV_LOG).readlines()[-40:]))
    raise RuntimeError("ollama serve did not come up")

# Confirm Ollama found the GPUs. If this says CPU, everything below will crawl.
print("  " + "\n  ".join(l.rstrip()[:160] for l in open(SRV_LOG)
                         if any(k in l for k in ("inference compute", "library=cuda",
                                                 "gpu", "GPU", "total blobs")))[:2000])

In [ ]:
# ============================================================
#  3. Get the model into the store.
# ============================================================

# We pull Ollama's own build rather than importing a GGUF from Hugging Face.
#
# WHY: `ollama create` from an Unsloth "UD" (Unsloth Dynamic) GGUF fails with
#   Error: failed to validate GGUF with llama-quantize without compatibility patches
# UD quants give each layer a different bit width, and Ollama's import-time
# llama-quantize validation gate rejects that. Same error is open upstream as
# ollama/ollama#17279 on UD-IQ2_M. There is no bypass flag.
#
# The registry build has no import step and therefore no gate. It is 18 GB
# rather than 16.5 GB, which still fits the 20 GB cache with room to spare.

MF = "/kaggle/tmp/Modelfile"

if BUILD_CACHE or not HAVE_CACHE:
    step(f"pulling {MODEL_TAG} from Ollama's registry (~18 GB)")
    sh([OLLAMA, "pull", MODEL_TAG], env=env)

    # Derive our own tag carrying the run parameters. FROM <model> references
    # existing layers rather than importing a file, so it costs no extra disk
    # and does not go through the llama-quantize gate that just failed.
    # Sampling values are the ones Qwen publishes for non-thinking mode; any
    # API request can override them per call.
    lines = [f"FROM {MODEL_TAG}",
             f"PARAMETER num_ctx {N_CTX}",
             "PARAMETER temperature 0.7",
             "PARAMETER top_p 0.8",
             "PARAMETER top_k 20",
             "PARAMETER min_p 0.0"]
    open(MF, "w").write(chr(10).join(lines) + chr(10))
    try:
        sh([OLLAMA, "create", MODEL_NAME, "-f", MF], env=env)
    except subprocess.CalledProcessError:
        # Should not happen for FROM <model>, but do not lose an 18 GB pull to it.
        step("  parameterised create failed; falling back to a plain alias")
        step("  (N_CTX still applies via the OLLAMA_CONTEXT_LENGTH env var)")
        sh([OLLAMA, "cp", MODEL_TAG, MODEL_NAME], env=env)

if BUILD_CACHE:
    size = int(subprocess.run(["du", "-sb", "/kaggle/working"],
                              capture_output=True, text=True).stdout.split()[0])
    step(f"/kaggle/working = {size/2**30:.2f} GiB   (Kaggle's cap is 20)")
    if size > 19 * 2**30:
        print("  WARNING: too close to the 20 GB output cap. Save Version may fail.")
        print("  Delete /kaggle/working/*.log and retry, or serve without a cache.")

    CACHE_DONE = True
    print("\n" + "=" * 70)
    print("  CACHE BUILT. Next:")
    print("    1. Save Version -> Save & Run All   (publishes /kaggle/working)")
    print("    2. wait for it to finish, then Stop Session")
    print("    3. set BUILD_CACHE = False")
    print("    4. Add Input -> Notebook Output -> this notebook; set CACHE")
    print("=" * 70)

step("models available:")
print(subprocess.run([OLLAMA, "list"], env=env, capture_output=True, text=True).stdout)

In [ ]:
# ============================================================
#  4. Auth proxy. Ollama has NO authentication of its own, and the
#     next cell puts it on the public internet.
# ============================================================

# SystemExit does not stop 'Run All' in Jupyter, so the guard is explicit.
if CACHE_DONE:
    raise RuntimeError("Cache build finished. Save Version, set BUILD_CACHE=False, "
                       "attach the output, then rerun.")

import secrets, threading, http.server, socketserver, urllib.request, urllib.error

PORT, API_KEY = 8000, secrets.token_urlsafe(24)
UPSTREAM = "http://127.0.0.1:11434"
HOP = {"connection", "keep-alive", "transfer-encoding", "content-length", "host"}

class Proxy(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, fmt, *a):
        # BaseHTTPRequestHandler calls this from log_request AND log_error with
        # different arities, so index nothing.
        print(f"    proxy {self.command} {self.path}", flush=True)

    def _authed(self):
        if self.headers.get("Authorization", "") == f"Bearer {API_KEY}":
            return True
        self.send_response(401)
        self.send_header("Content-Length", "0")
        self.end_headers()
        return False

    def _forward(self, body=None):
        if not self._authed():
            return
        req = urllib.request.Request(UPSTREAM + self.path, data=body, method=self.command)
        for k, v in self.headers.items():
            if k.lower() not in HOP and k.lower() != "authorization":
                req.add_header(k, v)
        try:
            up = urllib.request.urlopen(req, timeout=900)
        except urllib.error.HTTPError as e:
            up = e
        except Exception as e:
            self.send_response(502); self.send_header("Content-Length", "0"); self.end_headers()
            print(f"    proxy upstream error: {e}", flush=True)
            return
        self.send_response(up.status)
        for k, v in up.headers.items():
            if k.lower() not in HOP:
                self.send_header(k, v)
        # Chunked, so streamed completions arrive token by token instead of
        # being buffered until the request completes.
        self.send_header("Transfer-Encoding", "chunked")
        self.end_headers()
        try:
            while True:
                # readline, not read(n): read(n) blocks until n bytes arrive,
                # which would defeat streaming. SSE is line delimited.
                chunk = up.readline()
                if not chunk:
                    break
                self.wfile.write(b"%x\r\n" % len(chunk) + chunk + b"\r\n")
                self.wfile.flush()
            self.wfile.write(b"0\r\n\r\n")
            self.wfile.flush()
        except (BrokenPipeError, ConnectionResetError):
            pass

    def do_GET(self):
        self._forward()

    def do_POST(self):
        n = int(self.headers.get("Content-Length") or 0)
        self._forward(self.rfile.read(n) if n else b"")

class Threaded(socketserver.ThreadingMixIn, http.server.HTTPServer):
    daemon_threads = True
    allow_reuse_address = True

proxy = Threaded(("127.0.0.1", PORT), Proxy)
threading.Thread(target=proxy.serve_forever, daemon=True).start()
step(f"auth proxy listening on {PORT}, forwarding to {UPSTREAM}")

In [ ]:
# ============================================================
#  5. Public tunnel, then a best-effort test through it.
# ============================================================

if CACHE_DONE:
    raise RuntimeError("cache build run - stop here")

import re, socket, requests

if not shutil.which("cloudflared"):
    step("installing cloudflared")
    sh(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
    os.chmod("/usr/local/bin/cloudflared", 0o755)

# Load the model over localhost FIRST. Cloudflare's edge will drop a request
# that sits silent for minutes, and the first call has to read 18 GB onto the
# GPUs. Warming locally means the tunnel's first request is already fast.
step("warming the model on localhost - reads 18 GB onto the GPUs, several minutes")
w0 = time.time()
# max_tokens must be generous: Qwen3.8 is a thinking model and its reasoning
# tokens are billed against the budget but stripped from `content`. A small
# budget returns finish_reason="length" with an EMPTY string, which looks like
# a broken model. Never go below ~200 for this model.
warm = requests.post("http://127.0.0.1:11434/v1/chat/completions",
                     json={"model": MODEL_NAME, "max_tokens": 400, "temperature": 0,
                           "messages": [{"role": "user", "content": "hi"}]},
                     timeout=2400)
if not warm.ok:
    print(warm.text[:800]); print("".join(open(SRV_LOG).readlines()[-30:]))
    raise RuntimeError(f"model failed to load ({warm.status_code})")
step(f"model loaded and generating in {(time.time()-w0)/60:.1f} min")

# Where did the layers actually land? If this shows CPU, throughput will be unusable.
print(subprocess.run([OLLAMA, "ps"], env=env, capture_output=True, text=True).stdout)

TUN_LOG = "/kaggle/working/tunnel.log"
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=open(TUN_LOG, "w"), stderr=subprocess.STDOUT)

PUBLIC = None
for _ in range(40):
    time.sleep(3)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(TUN_LOG).read())
    if m:
        PUBLIC = m.group(0); break
assert PUBLIC, f"tunnel failed, see {TUN_LOG}"
step(f"tunnel up: {PUBLIC}")

# --- best effort self-test ---------------------------------------------------
# Kaggle's resolver often cannot see a brand new trycloudflare hostname, while
# an ordinary laptop resolves it immediately. So this test failing proves
# nothing, and must never abort the run. Your laptop is the client that counts.
host, verdict = PUBLIC.split("//", 1)[1], "not attempted"
for i in range(10):
    try:
        socket.getaddrinfo(host, 443); break
    except socket.gaierror:
        if i == 0:
            step("Kaggle cannot resolve the tunnel hostname yet, retrying for ~90s")
        time.sleep(10)

try:
    r = requests.post(PUBLIC + "/v1/chat/completions",
                      headers={"Authorization": f"Bearer {API_KEY}"},
                      json={"model": MODEL_NAME, "max_tokens": 400, "temperature": 0,
                            "messages": [{"role": "user", "content": "Reply with exactly: ready"}]},
                      timeout=300)
    unauth = requests.get(PUBLIC + "/v1/models", timeout=60).status_code
    verdict = (f"reply {r.json()['choices'][0]['message']['content'].strip()[:40]!r}, "
               f"unauthenticated request got {unauth} "
               + ("(correctly rejected)" if unauth == 401 else "*** NOT PROTECTED ***")) \
              if r.ok else f"HTTP {r.status_code}: {r.text[:200]}"
except Exception as e:
    verdict = (f"could not test from inside Kaggle ({type(e).__name__}). "
               "This is usually just Kaggle's DNS, not a broken tunnel. "
               "Test from your laptop.")

print("\n" + "=" * 70)
print("  ENDPOINT PUBLISHED")
print("=" * 70)
print(f"  OPENAI_BASE_URL = {PUBLIC}/v1")
print(f"  OPENAI_API_KEY  = {API_KEY}")
print(f"  model           = {MODEL_NAME}")
print(f"  self-test       = {verdict}")
print(f"  setup took {(time.time()-T0)/60:.1f} min")
print("=" * 70)
print("  Verify from your laptop:")
print(f"    curl -s {PUBLIC}/v1/chat/completions \\")
print(f"      -H 'Authorization: Bearer {API_KEY}' -H 'Content-Type: application/json' \\")
print("      -d '{\"model\":\"" + MODEL_NAME + "\",\"messages\":[{\"role\":\"user\",\"content\":\"hi\"}],\"max_tokens\":16}'")
print("\n  URL and key are new every session. Put them in a .env, never in git.")

In [ ]:
# ============================================================
#  6. Hold the endpoint open, then stop the model automatically.
#     Interrupt this cell to stop sooner.
# ============================================================

if CACHE_DONE:
    raise RuntimeError("cache build run - stop here")

end = time.time() + RUN_MINUTES * 60
step(f"serving until {datetime.datetime.fromtimestamp(end):%H:%M:%S}")
try:
    while time.time() < end:
        if serve.poll() is not None:
            print("ollama serve died:")
            print("".join(open(SRV_LOG).readlines()[-30:]))
            break
        step(f"alive, {int((end-time.time())/60)} min left  |  {PUBLIC}")
        time.sleep(300)
finally:
    for p in (tunnel, serve):
        try: p.terminate()
        except Exception: pass
    try: proxy.shutdown()
    except Exception: pass
    time.sleep(5)
    print("\n" + "=" * 70)
    print("  Model stopped. GPU memory is free. Tunnel is closed.")
    print("")
    print("  YOUR QUOTA IS STILL RUNNING.")
    print("  Kaggle charges for an ATTACHED GPU session, not for GPU work.")
    print("  Click STOP SESSION in the sidebar. That is the only thing")
    print("  that stops the meter.")
    print("=" * 70)

## What actually stops the quota

| Action | Frees GPU memory | **Stops quota** |
|---|---|---|
| Timer expires, or you interrupt the cell | Yes | **No** |
| Closing the browser tab | No | **No** |
| **Stop Session** in the sidebar | Yes | **Yes** |
| Kaggle's idle timeout | Yes | Yes, eventually |

Kaggle bills for a GPU session being **attached**, not for the GPU doing work. The 30 hours per week
drain at the same rate whether the model is generating or sitting idle.

## Calling it from the laptop

```python
from openai import OpenAI
client = OpenAI(base_url="<OPENAI_BASE_URL>", api_key="<OPENAI_API_KEY>")
print(client.chat.completions.create(
    model="qwen",
    messages=[{"role": "user", "content": "hello"}],
).choices[0].message.content)
```

Both values change every session, because the Cloudflare quick tunnel gets a new random hostname and
the key is regenerated each run. Write them into a local `.env`, which `.gitignore` already excludes.

## If it fails

| Symptom | Cause and fix |
|---|---|
| `wget` 404 on the tarball | Ollama moved or yanked the pinned release. Bump `OLLAMA_VER` |
| Log says `library=cpu`, generation crawls | Ollama did not see the GPUs. Check the probe cell showed two |
| Out of memory on load | Drop `N_CTX` to 4096. The registry has no smaller tag than `27b` |
| `Error: unknown architecture` | Ollama is older than v0.32.12. Raise `OLLAMA_VER` |
| Only one GPU listed | Accelerator is not set to **T4 x2** |
| Save Version fails on the build run | `/kaggle/working` exceeded 20 GB. Delete the `.log` files and retry |
| 401 from your own client | The key is regenerated each session. Copy the new one |
| Replies come back as an empty string | `max_tokens` too low. Qwen3.8's reasoning tokens count against the budget but are stripped from `content`. Use 200+ |
| `failed to validate GGUF with llama-quantize` | You pointed it at a GGUF file. Use the registry tag |
| HF download 401s | Secret not attached. Add-ons -> Secrets -> toggle `HF_TOKEN` on |
| URL stops working | Session ended. Run All again for a new URL and key |
| `\"qwen\" does not support chat` + `total blobs: 0` | Cache has manifests but no weights. You Quick Saved from a serving session. Rebuild with `BUILD_CACHE=True` and save from that run only |
| `NameResolutionError` on `*.trycloudflare.com` | Kaggle's resolver cannot see the new hostname. Harmless. Test from your laptop, which can |

## Making it faster

Decode is memory-bandwidth bound. One token requires reading all 18 GB of weights, a T4 gives about
320 GB/s, and layer split runs the two cards **sequentially rather than in parallel**, so the ceiling
is roughly `320 / 18 = 17.8 tok/s`. The measured 9.2 is about half of that, which is normal once
attention overhead and the PCIe hop at each layer boundary are counted. The second T4 buys capacity to
hold the model, not speed.

So there are only two ways to go faster: read fewer bytes per token, or emit more tokens per read.

| Lever | Gain | Quality cost |
|---|---|---|
| `MODEL_TAG = "qwen3.8:27b-mtp-q4_K_M"` | +33 to 39% reported | **None**, lossless |
| `think: false` on the native `/api/chat` | ~40 to 50% less wall time | Some reasoning depth |
| Design prompts to emit less | Large | None |
| Streaming | Perceived only | None, already on |
| Smaller quant | Linear with size | Real |

Not available on this hardware: **tensor parallelism** (Ollama only does layer split; vLLM has it but
is broken on Turing) and **MoE** (Qwen3.8 ships dense only).

Note that thinking can only be disabled on the **native** `/api/chat`, not on `/v1/chat/completions`.
The auth proxy forwards every path, so `/api/chat` already works through the tunnel unchanged.

## Honest caveats

- **Measured on 2x T4, 31 Aug 2026.** Model loads 100% on GPU at 8192 context. Cold start from cache
  is ~2 min, versus ~20 min without. Sustained generation is **~9.2 tok/s** over an 842 token
  response. Streaming arrives incrementally through the tunnel. JSON mode returns schema-valid output.
- **Qwen3.8 is a thinking model.** Reasoning tokens are billed against `max_tokens` but stripped from
  `content`, so a small budget yields `finish_reason="length"` and an empty string. A two word answer
  cost 37 completion tokens. Budget 200+ on every call.
- **Cloudflare quick tunnels are not a production ingress.** They are rate limited, unauthenticated
  at the tunnel layer, and the hostname is public. The bearer proxy is what keeps the endpoint from
  being open to anyone who stumbles on the URL. Do not treat the URL itself as a secret.
- **No personal health data should ever cross this endpoint** while it is on a public tunnel. During
  development the agent sends synthetic or public-dataset records. That constraint is in the
  proposal, not an afterthought here.
- **Ollama is slower than vLLM under concurrency.** Irrelevant here: one user, occasional calls.
- **Symlinked blobs assume the cached dataset stays attached** for the whole session. It does.
- **The registry build is 18 GB against a 20 GB cache cap**, about 2 GB of headroom. If Ollama
  republishes `qwen3.8:27b` at a larger quant, the cache build is what breaks, and it tells you so
  before you reach Save Version.

In [ ]:
# OPTIONAL FALLBACK: end the container from inside the notebook.
# Kills every process, which terminates the session and stops the quota.
# Blunt: anything unsaved in /kaggle/working is lost. Prefer Stop Session.

CONFIRM = False   # set True, then run this cell

if CONFIRM:
    import signal
    print("terminating container...")
    os.kill(1, signal.SIGKILL)
else:
    print("Not armed. Set CONFIRM = True to terminate.")
    print("Prefer Stop Session in the sidebar.")